# M08: Feature Engineering
This notebook applies the feature engineering pipeline defined in `src/features.py`.

### Steps:
1. **Encode target**: Convert `Churn` from Yes/No to 1/0.
2. **Drop customerID**: Not a predictive feature.
3. **Engineer new features**:
   - `AvgChargesPerMonth` = TotalCharges / max(tenure, 1) — normalizes spending by tenure.
   - `tenure_group` — buckets tenure into interpretable segments.
4. **One-hot encode** all remaining categorical columns (drop_first to avoid multicollinearity).
5. **Standard scale** numerical columns for models sensitive to feature magnitude (e.g., SVM, KNN, Logistic Regression).

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data import load_raw_data
from src.cleaning import clean_data
from src.features import build_features, add_engineered_features, encode_target, drop_customer_id

df = clean_data(load_raw_data("../data/raw"))
print(f"Clean data shape: {df.shape}")


Clean data shape: (7043, 21)


## Engineered Features Preview

In [2]:
df_eng = add_engineered_features(df)
df_eng[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgChargesPerMonth', 'tenure_group']].head(10)

,customerID,tenure,MonthlyCharges,TotalCharges,AvgChargesPerMonth,tenure_group
0,7590-VHVEG,1,29.85,29.85,29.850000,0-12
1,5575-GNVDE,34,56.95,1889.50,55.573529,25-48
2,3668-QPYBK,2,53.85,108.15,54.075000,0-12
3,7795-CFOCW,45,42.30,1840.75,40.905556,25-48
4,9237-HQITU,2,70.70,151.65,75.825000,0-12
5,9305-CDSKC,8,99.65,820.50,102.562500,0-12
6,1452-KIOVK,22,89.10,1949.40,88.609091,13-24
7,6713-OKOMC,10,29.75,301.90,30.190000,0-12
8,7892-POOKP,28,104.80,3046.05,108.787500,25-48
9,6388-TABGU,62,56.15,3487.95,56.257258,61-72


## Full Pipeline Output (unscaled, for inspection)

In [3]:
df_feat = build_features(df, scale=False)
print(f"Feature-engineered shape: {df_feat.shape}")
print(f"\nColumns ({len(df_feat.columns)}):")
print(df_feat.columns.tolist())
print(f"\nNull count: {df_feat.isnull().sum().sum()}")
print(f"\nObject columns remaining: {df_feat.select_dtypes(include=['object']).columns.tolist()}")
df_feat.head()

Feature-engineered shape: (7043, 36)

Columns (36):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'AvgChargesPerMonth', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'tenure_group_13-24', 'tenure_group_25-48', 'tenure_group_49-60', 'tenure_group_61-72']

Null count: 0

Object columns remaining: []


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,AvgChargesPerMonth,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_13-24,tenure_group_25-48,tenure_group_49-60,tenure_group_61-72
0,0,1,29.85,29.85,0,29.850000,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
1,0,34,56.95,1889.50,0,55.573529,1,0,0,1,...,1,0,0,0,0,1,0,1,0,0
2,0,2,53.85,108.15,1,54.075000,1,0,0,1,...,0,0,1,0,0,1,0,0,0,0
3,0,45,42.30,1840.75,0,40.905556,1,0,0,0,...,1,0,0,0,0,0,0,1,0,0
4,0,2,70.70,151.65,1,75.825000,0,0,0,1,...,0,0,1,0,1,0,0,0,0,0


## Save processed data for modeling

In [4]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_feat.to_csv("../data/processed/telco_features.csv", index=False)
print("Saved to data/processed/telco_features.csv")

Saved to data/processed/telco_features.csv
